In [10]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

BASE_DIR = Path(".")
CATALOG_PATH = BASE_DIR / "recommendation" / "recommendation_items.csv"

# Giữ đúng thứ tự này cho cả Student Profile và Item Profile.
CONTENT_FEATURES = [
    "low_grade",
    "low_pass_rate",
    "failed_burden",
    "evaluation_issue",
    "workload_gap",
    "high_performance",
]

RAW_FEATURES = [
    "1st_sem_enrolled",
    "1st_sem_evaluations",
    "1st_sem_without_evaluations",
    "1st_sem_approved",
    "1st_sem_failed",
    "1st_sem_grade",
    "1st_sem_pass_rate",
]

MAX_GRADE = 20.0
DEFAULT_TOP_N = 3

print("CONTENT_FEATURES:", CONTENT_FEATURES)


CONTENT_FEATURES: ['low_grade', 'low_pass_rate', 'failed_burden', 'evaluation_issue', 'workload_gap', 'high_performance']


In [11]:
def load_catalog(path=CATALOG_PATH):
    """Load và kiểm tra recommendation catalog."""
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy catalog: {path.resolve()}")

    catalog = pd.read_csv(path, encoding="utf-8-sig")

    required_columns = {
        "item_id", "item_name", "category", "description", "action_plan",
        *CONTENT_FEATURES,
    }
    missing = required_columns - set(catalog.columns)
    if missing:
        raise ValueError(f"Catalog thiếu cột: {sorted(missing)}")

    if catalog["item_id"].isna().any():
        raise ValueError("Catalog có item_id bị thiếu.")

    if catalog["item_id"].duplicated().any():
        dup = catalog.loc[catalog["item_id"].duplicated(), "item_id"].tolist()
        raise ValueError(f"item_id bị trùng: {dup}")

    for col in CONTENT_FEATURES:
        catalog[col] = pd.to_numeric(catalog[col], errors="raise")
        if ((catalog[col] < 0) | (catalog[col] > 1)).any():
            bad = catalog.loc[(catalog[col] < 0) | (catalog[col] > 1), ["item_id", col]]
            raise ValueError(f"Giá trị {col} ngoài [0,1]:\n{bad}")

    if len(catalog) == 0:
        raise ValueError("Catalog đang rỗng.")

    return catalog.reset_index(drop=True)


catalog_df = load_catalog()

print(f"Load catalog thành công: {len(catalog_df)} giải pháp")
display(catalog_df[["item_id", "item_name", "category", *CONTENT_FEATURES]])


Load catalog thành công: 10 giải pháp


,item_id,item_name,category,low_grade,low_pass_rate,failed_burden,evaluation_issue,workload_gap,high_performance
0,REC_001,Học kèm / Tutoring,Academic Support,0.9,0.9,0.8,0.2,0.4,0.0
1,REC_002,Học lại học phần chưa đạt,Course Recovery,0.6,0.9,1.0,0.1,0.7,0.0
2,REC_003,Điều chỉnh khối lượng học tập,Workload Management,0.5,0.7,0.8,0.1,1.0,0.0
3,REC_004,Tư vấn Cố vấn học tập,Academic Advising,0.4,0.5,0.5,1.0,0.3,0.0
4,REC_005,Lập kế hoạch tự học cá nhân,Study Planning,0.5,0.6,0.4,0.4,0.7,0.1
5,REC_006,Tham gia nhóm học tập,Peer Learning,0.6,0.7,0.5,0.3,0.5,0.1
6,REC_007,Theo dõi tiến độ định kỳ,Progress Monitoring,0.3,0.4,0.3,0.5,0.4,0.2
7,REC_008,Duy trì lộ trình chuẩn,Study Maintenance,0.1,0.1,0.1,0.0,0.2,0.8
8,REC_009,Hoạt động học thuật nâng cao,Academic Enrichment,0.0,0.0,0.0,0.0,0.1,1.0
9,REC_010,Ôn tập và củng cố học phần yếu,Remedial Study,0.8,0.8,0.7,0.4,0.5,0.0


In [12]:
def validate_student_data(student_df, require_pass_rate=False):
    """Kiểm tra dữ liệu đầu vào của một hoặc nhiều sinh viên."""
    missing = set(RAW_FEATURES) - set(student_df.columns)
    if missing:
        raise ValueError(f"Student data thiếu cột: {sorted(missing)}")

    df = student_df.copy()
    for col in RAW_FEATURES:
        df[col] = pd.to_numeric(df[col], errors="raise")

    non_negative_cols = [
        "1st_sem_enrolled",
        "1st_sem_evaluations",
        "1st_sem_without_evaluations",
        "1st_sem_approved",
        "1st_sem_failed",
        "1st_sem_grade",
    ]
    for col in non_negative_cols:
        if (df[col] < 0).any():
            raise ValueError(f"Cột {col} chứa giá trị âm.")

    if (df["1st_sem_grade"] > MAX_GRADE).any():
        raise ValueError(f"1st_sem_grade phải nằm trong khoảng 0–{MAX_GRADE}.")

    if require_pass_rate and ((df["1st_sem_pass_rate"] < 0) | (df["1st_sem_pass_rate"] > 1)).any():
        raise ValueError("1st_sem_pass_rate phải nằm trong [0,1].")

    return df


def add_derived_features(student_df):
    """Tự động tính pass_rate nếu cần và tạo các tỷ lệ trung gian."""
    df = student_df.copy()

    df["1st_sem_pass_rate"] = np.where(
        df["1st_sem_evaluations"] > 0,
        df["1st_sem_approved"] / df["1st_sem_evaluations"],
        0.0,
    )
    df["1st_sem_pass_rate"] = df["1st_sem_pass_rate"].clip(0, 1)

    enrolled = df["1st_sem_enrolled"].replace(0, np.nan)
    df["failed_burden"] = (df["1st_sem_failed"] / enrolled).fillna(0).clip(0, 1)
    df["evaluation_issue"] = (df["1st_sem_without_evaluations"] / enrolled).fillna(0).clip(0, 1)
    return df


In [13]:
def create_student_content_profile(student_df):
    """Chuyển 7 raw features thành 6D semantic Student Content Profile trong [0,1]."""
    df = add_derived_features(student_df)
    validate_student_data(df, require_pass_rate=True)

    grade_norm = (df["1st_sem_grade"] / MAX_GRADE).clip(0, 1)
    enrolled = df["1st_sem_enrolled"].replace(0, np.nan)

    semantic = pd.DataFrame(index=df.index)
    semantic["low_grade"] = (1.0 - grade_norm).clip(0, 1)
    semantic["low_pass_rate"] = (1.0 - df["1st_sem_pass_rate"]).clip(0, 1)
    semantic["failed_burden"] = (df["1st_sem_failed"] / enrolled).fillna(0).clip(0, 1)
    semantic["evaluation_issue"] = (
        df["1st_sem_without_evaluations"] / enrolled
    ).fillna(0).clip(0, 1)
    semantic["workload_gap"] = (
        1.0 - (df["1st_sem_approved"] / enrolled).fillna(0)
    ).clip(0, 1)
    semantic["high_performance"] = (
        grade_norm * df["1st_sem_pass_rate"]
    ).clip(0, 1)

    return semantic[CONTENT_FEATURES]


print("Các semantic features đã định nghĩa:")
for feature in CONTENT_FEATURES:
    print(f"- {feature}")


Các semantic features đã định nghĩa:
- low_grade
- low_pass_rate
- failed_burden
- evaluation_issue
- workload_gap
- high_performance


## Công thức Student Content Profile

- `low_grade = 1 - grade / 20`
- `low_pass_rate = 1 - pass_rate`
- `failed_burden = failed / enrolled`
- `evaluation_issue = without_evaluations / enrolled`
- `workload_gap = 1 - approved / enrolled`
- `high_performance = (grade / 20) × pass_rate`

Các giá trị đều được giới hạn về `[0,1]` để tương thích trực tiếp với Item Profile trong catalog.

> `workload_gap` không được diễn giải là bằng chứng sinh viên "bị quá tải"; nó chỉ biểu diễn khoảng cách giữa khối lượng đăng ký và kết quả đạt được.


In [14]:
def get_item_matrix(catalog):
    """Lấy Item Profile theo đúng thứ tự CONTENT_FEATURES."""
    return catalog[CONTENT_FEATURES].to_numpy(dtype=float)


def explain_similarity(student_vector, item_vector):
    """Phân rã đóng góp của từng chiều vào cosine similarity.

    Vì tất cả giá trị đều không âm, contribution_i = s_i*i_i/(||s||*||i||)
    và tổng các contribution bằng cosine similarity.
    """
    s = np.asarray(student_vector, dtype=float).reshape(-1)
    i = np.asarray(item_vector, dtype=float).reshape(-1)

    s_norm = np.linalg.norm(s)
    i_norm = np.linalg.norm(i)
    if s_norm == 0 or i_norm == 0:
        return pd.Series(0.0, index=CONTENT_FEATURES)

    contributions = (s * i) / (s_norm * i_norm)
    return pd.Series(contributions, index=CONTENT_FEATURES)


def recommend_study_plans(student_data, catalog=catalog_df, top_n=3, low_confidence_threshold=None):
    """CBF recommendation bằng Cosine Similarity.

    low_confidence_threshold=None: không ép ngưỡng; luôn trả Top-N.
    Nếu đặt ngưỡng, chỉ gắn cờ low_confidence chứ không thay recommendation bằng rule cố định.
    """
    if top_n < 1:
        raise ValueError("top_n phải >= 1")

    if len(student_data) == 0:
        raise ValueError("student_data đang rỗng.")

    student_profile = create_student_content_profile(student_data)
    student_vectors = student_profile[CONTENT_FEATURES].to_numpy(dtype=float)
    item_vectors = get_item_matrix(catalog)

    similarities = cosine_similarity(student_vectors, item_vectors)
    rows = []

    for student_idx in range(len(student_profile)):
        order = np.argsort(-similarities[student_idx], kind="stable")[:top_n]
        best_score = float(similarities[student_idx, order[0]])

        for rank, item_idx in enumerate(order, start=1):
            item = catalog.iloc[item_idx]
            contribution = explain_similarity(
                student_vectors[student_idx],
                item_vectors[item_idx],
            )
            top_dimensions = contribution.sort_values(ascending=False).head(3)

            rows.append({
                "student_index": student_idx,
                "rank": rank,
                "item_id": item["item_id"],
                "item_name": item["item_name"],
                "category": item["category"],
                "description": item["description"],
                "action_plan": item["action_plan"],
                "similarity_score": float(similarities[student_idx, item_idx]),
                "low_confidence": (
                    low_confidence_threshold is not None
                    and best_score < low_confidence_threshold
                ),
                "top_matching_dimensions": ", ".join(top_dimensions.index),
            })

    return pd.DataFrame(rows), student_profile


In [15]:
# TEST 1 — Sinh viên rớt nhiều môn
student_1 = pd.DataFrame([{
    "1st_sem_enrolled": 6,
    "1st_sem_evaluations": 6,
    "1st_sem_without_evaluations": 0,
    "1st_sem_approved": 2,
    "1st_sem_failed": 4,
    "1st_sem_grade": 9.5,
}])

# TEST 2 — Nhiều học phần không có kết quả đánh giá
student_2 = pd.DataFrame([{
    "1st_sem_enrolled": 5,
    "1st_sem_evaluations": 2,
    "1st_sem_without_evaluations": 3,
    "1st_sem_approved": 1,
    "1st_sem_failed": 1,
    "1st_sem_grade": 10.0,
}])

# TEST 3 — Thành tích tốt
student_3 = pd.DataFrame([{
    "1st_sem_enrolled": 6,
    "1st_sem_evaluations": 6,
    "1st_sem_without_evaluations": 0,
    "1st_sem_approved": 6,
    "1st_sem_failed": 0,
    "1st_sem_grade": 16.5,
}])

# TEST 4 — Workload gap cao
student_4 = pd.DataFrame([{
    "1st_sem_enrolled": 8,
    "1st_sem_evaluations": 8,
    "1st_sem_without_evaluations": 0,
    "1st_sem_approved": 3,
    "1st_sem_failed": 5,
    "1st_sem_grade": 11.0,
}])

# TEST 5 — Điểm khá, ít failed nhưng vẫn cần theo dõi nhẹ
student_5 = pd.DataFrame([{
    "1st_sem_enrolled": 6,
    "1st_sem_evaluations": 6,
    "1st_sem_without_evaluations": 0,
    "1st_sem_approved": 5,
    "1st_sem_failed": 1,
    "1st_sem_grade": 13.0,
}])


test_cases = {
    "Học yếu, rớt nhiều môn": student_1,
    "Evaluation issue cao": student_2,
    "Thành tích tốt": student_3,
    "Workload gap cao": student_4,
    "Kết quả khá, cần hỗ trợ nhẹ": student_5,
}

for profile_name, student in test_cases.items():
    recommendations, profile = recommend_study_plans(
        student,
        catalog=catalog_df,
        top_n=3,
        low_confidence_threshold=None,
    )
    print(f"\n{'=' * 70}\nHỒ SƠ: {profile_name}\n{'=' * 70}")
    print("Student Content Profile:")
    display(profile.round(3))
    print("Top 3 recommendations:")
    display(recommendations[[
        "rank", "item_id", "item_name", "category",
        "similarity_score", "top_matching_dimensions"
    ]].round({"similarity_score": 4}))



HỒ SƠ: Học yếu, rớt nhiều môn
Student Content Profile:


,low_grade,low_pass_rate,failed_burden,evaluation_issue,workload_gap,high_performance
0,0.525,0.667,0.667,0.0,0.667,0.158


Top 3 recommendations:


,rank,item_id,item_name,category,similarity_score,top_matching_dimensions
0,1,REC_002,Học lại học phần chưa đạt,Course Recovery,0.9807,"failed_burden, low_pass_rate, workload_gap"
1,2,REC_003,Điều chỉnh khối lượng học tập,Workload Management,0.9762,"workload_gap, failed_burden, low_pass_rate"
2,3,REC_006,Tham gia nhóm học tập,Peer Learning,0.9512,"low_pass_rate, workload_gap, failed_burden"



HỒ SƠ: Evaluation issue cao
Student Content Profile:


...

,rank,item_id,item_name,category,similarity_score,top_matching_dimensions
0,1,REC_002,Học lại học phần chưa đạt,Course Recovery,0.9743,"failed_burden, low_pass_rate, workload_gap"
1,2,REC_003,Điều chỉnh khối lượng học tập,Workload Management,0.9713,"workload_gap, failed_burden, low_pass_rate"
2,3,REC_006,Tham gia nhóm học tập,Peer Learning,0.9440,"low_pass_rate, failed_burden, workload_gap"



HỒ SƠ: Kết quả khá, cần hỗ trợ nhẹ
Student Content Profile:


,low_grade,low_pass_rate,failed_burden,evaluation_issue,workload_gap,high_performance
0,0.35,0.167,0.167,0.0,0.167,0.542


Top 3 recommendations:


,rank,item_id,item_name,category,similarity_score,top_matching_dimensions
0,1,REC_008,Duy trì lộ trình chuẩn,Study Maintenance,0.8986,"high_performance, low_grade, workload_gap"
1,2,REC_009,Hoạt động học thuật nâng cao,Academic Enrichment,0.7863,"high_performance, workload_gap, low_pass_rate"
2,3,REC_006,Tham gia nhóm học tập,Peer Learning,0.6435,"low_grade, low_pass_rate, failed_burden"


In [16]:
# Kiểm tra tự động các bất biến quan trọng của CBF
profile_all = pd.concat(
    [create_student_content_profile(df) for df in test_cases.values()],
    ignore_index=True,
)

assert profile_all.shape[1] == 6
assert profile_all.columns.tolist() == CONTENT_FEATURES
assert np.isfinite(profile_all.to_numpy()).all()
assert (profile_all.to_numpy() >= 0).all()
assert (profile_all.to_numpy() <= 1).all()

item_matrix = get_item_matrix(catalog_df)
assert item_matrix.shape[1] == 6
assert np.isfinite(item_matrix).all()
assert (item_matrix >= 0).all()
assert (item_matrix <= 1).all()

# Similarity phải nằm trong [-1,1]; ở đây vector không âm nên thực tế nằm [0,1].
sim_test = cosine_similarity(profile_all.to_numpy(), item_matrix)
assert np.isfinite(sim_test).all()
assert (sim_test >= -1e-9).all()
assert (sim_test <= 1 + 1e-9).all()

print("PASS — Student Profile và Item Profile cùng không gian [0,1]^6.")
print("PASS — Cosine Similarity hợp lệ.")
print("PASS — Catalog không có item_id trùng và có đầy đủ 6 semantic features.")


PASS — Student Profile và Item Profile cùng không gian [0,1]^6.
PASS — Cosine Similarity hợp lệ.
PASS — Catalog không có item_id trùng và có đầy đủ 6 semantic features.


In [17]:
# Kiểm tra trường hợp similarity thấp mà không dùng fallback hard-code
low_match_student = pd.DataFrame([{
    "1st_sem_enrolled": 4,
    "1st_sem_evaluations": 4,
    "1st_sem_without_evaluations": 0,
    "1st_sem_approved": 2,
    "1st_sem_failed": 2,
    "1st_sem_grade": 10.5,
}])

low_match_result, _ = recommend_study_plans(
    low_match_student,
    catalog=catalog_df,
    top_n=3,
    low_confidence_threshold=0.90,
)

print("Nếu best similarity < 0.90, hệ thống chỉ gắn cờ low_confidence:")
display(low_match_result[[
    "rank", "item_id", "item_name", "similarity_score", "low_confidence"
]])


Nếu best similarity < 0.90, hệ thống chỉ gắn cờ low_confidence:


,rank,item_id,item_name,similarity_score,low_confidence
0,1,REC_002,Học lại học phần chưa đạt,0.949102,False
1,2,REC_006,Tham gia nhóm học tập,0.943585,False
2,3,REC_003,Điều chỉnh khối lượng học tập,0.941450,False


In [18]:
# Hàm tiện ích cho Streamlit / batch prediction
def recommend_one(student_dict, catalog=catalog_df, top_n=3):
    """Nhận dict của một sinh viên và trả Top-N recommendation."""
    student_df = pd.DataFrame([student_dict])
    result, profile = recommend_study_plans(
        student_df,
        catalog=catalog,
        top_n=top_n,
        low_confidence_threshold=None,
    )
    return result, profile.iloc[0]


def recommend_batch(student_df, catalog=catalog_df, top_n=3):
    """Chạy CBF cho nhiều sinh viên. Kết quả có student_index để nối với dữ liệu gốc."""
    result, profiles = recommend_study_plans(
        student_df,
        catalog=catalog,
        top_n=top_n,
        low_confidence_threshold=None,
    )
    return result, profiles


example_dict = {
    "1st_sem_enrolled": 6,
    "1st_sem_evaluations": 6,
    "1st_sem_without_evaluations": 1,
    "1st_sem_approved": 3,
    "1st_sem_failed": 3,
    "1st_sem_grade": 10.5,
}

demo_result, demo_profile = recommend_one(example_dict)
print("Hồ sơ semantic:")
display(demo_profile.to_frame("value").round(3))
print("Recommendations:")
display(demo_result[[
    "rank", "item_id", "item_name", "category",
    "similarity_score", "top_matching_dimensions"
]].round({"similarity_score": 4}))


Hồ sơ semantic:


,value
low_grade,0.475
low_pass_rate,0.500
failed_burden,0.500
evaluation_issue,0.167
workload_gap,0.500
high_performance,0.262


Recommendations:


,rank,item_id,item_name,category,similarity_score,top_matching_dimensions
0,1,REC_006,Tham gia nhóm học tập,Peer Learning,0.9714,"low_pass_rate, low_grade, failed_burden"
1,2,REC_005,Lập kế hoạch tự học cá nhân,Study Planning,0.9533,"workload_gap, low_pass_rate, low_grade"
2,3,REC_002,Học lại học phần chưa đạt,Course Recovery,0.9466,"failed_burden, low_pass_rate, workload_gap"
